In [1]:
# pip install opencv-contrib-python

In [2]:
import cv2
import numpy as np

In [3]:
image = cv2.imread("test_image.jpg")        # Reads an image from a file
cv2.imshow("Result",image)                 # Displays an image in a window.
cv2.waitKey(0)                           # Time in milliseconds to wait.

-1

In [ ]:
def canny(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel = 5
    blur = cv2.GaussianBlur(gray,(kernel, kernel),0)
    canny = cv2.Canny(gray, 50, 150)
    return canny

In [ ]:
def region_of_interest(canny):
    height = canny.shape[0]                         
    width = canny.shape[1]
    mask = np.zeros_like(canny)                 #Dark Image as fully zeros, all rows and columns equal to Canny(Frame/Img)
 
    triangle = np.array([[                      #Using Matplot Lib check the coordinate points
    (200, height),
    (550, 250),
    (1100, height),]], np.int32)
 
    cv2.fillPoly(mask, triangle, 255)           #Fills mask with polygon/triangle with White space
    masked_image = cv2.bitwise_and(canny, mask) 
    return masked_image

In [ ]:
def display_lines(img,lines):
    line_image = np.zeros_like(img)
    if lines is not None:
        for line in lines:
            for x1, y1, x2, y2 in line:
                cv2.line(line_image,(x1,y1),(x2,y2),(255,0,0),10)       #Draws lines with 2 points
                    #Drwing on line image, 1st point, 2nd point, Color, Thickness
    return line_image

In [ ]:
def make_points(image, line):
   slope, intercept = line
   y1 = int(image.shape[0]) # bottom of the image               #print(image.shape) - [704,1279,3]
   y2 = int(y1*3/5)         # slightly lower than the middle    # y1 = 704
   x1 = int((y1 - intercept)/slope)
   x2 = int((y2 - intercept)/slope)
   return [[x1, y1, x2, y2]]

def average_slope_intercept(image, lines):
    left_fit    = []            #Coordinates of the avg lines on left
    right_fit   = []            #Coordinates of the avg lines on right
    if lines is None:
        return None
    for line in lines:
        for x1, y1, x2, y2 in line:
            fit = np.polyfit((x1,x2), (y1,y2), 1)       #Fits a 1st degree polynomial
            slope = fit[0]
            intercept = fit[1]
            if slope < 0: # y is reversed in image
                left_fit.append((slope, intercept))
            else:
                right_fit.append((slope, intercept))
    # add more weight to longer lines
    left_fit_average  = np.average(left_fit, axis=0)
    right_fit_average = np.average(right_fit, axis=0)
    left_line  = make_points(image, left_fit_average)
    right_line = make_points(image, right_fit_average)
    averaged_lines = [left_line, right_line]
    return averaged_lines


In [ ]:
# image = cv2.imread('test_image.jpg')
# lane_image = np.copy(image)
# lane_canny = canny(lane_image)
# cropped_canny = region_of_interest(lane_canny)
# lines = cv2.HoughLinesP(cropped_canny, 2, np.pi/180, 100, np.array([]), minLineLength=40,maxLineGap=5)
# averaged_lines = average_slope_intercept(image, lines)
# line_image = display_lines(lane_image, averaged_lines)
# combo_image = cv2.addWeighted(lane_image, 0.8, line_image, 1, 0)

cap = cv2.VideoCapture("test2.mp4")
while(cap.isOpened()):
    _, frame = cap.read()       #Unpacks every video frame

    canny_image = canny(frame)                          #Edge Detection

    cropped_canny = region_of_interest(canny_image)     #ROI

    lines = cv2.HoughLinesP(cropped_canny, 2, np.pi/180, 100, np.array([]), minLineLength=40,maxLineGap=5)
                                #Pixels, 1 Radion(angle), No of intersectiones, Placeholder Array, Threshold, To connect each small line
    averaged_lines = average_slope_intercept(frame, lines)
    line_image = display_lines(frame, averaged_lines)
    combo_image = cv2.addWeighted(frame, 0.8, line_image, 1, 1)
                            #
    cv2.imshow("result", combo_image)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 